In [1]:
import pandas as pd

# Load raw files
orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
category_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

print("Files loaded!")

Files loaded!


In [2]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Date columns fixed!")
print(orders.dtypes)

Date columns fixed!
order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [3]:
# We only want completed orders for accurate analysis
orders_delivered = orders[orders['order_status'] == 'delivered'].copy()

print(f"Total orders:     {len(orders)}")
print(f"Delivered orders: {len(orders_delivered)}")
print(f"Removed:          {len(orders) - len(orders_delivered)} non-delivered orders")

Total orders:     99441
Delivered orders: 96478
Removed:          2963 non-delivered orders


In [4]:
# Some orders have multiple payment rows — sum them up per order
payments_agg = payments.groupby('order_id')['payment_value'].sum().reset_index()
payments_agg.rename(columns={'payment_value': 'total_payment'}, inplace=True)

print(f"Payments aggregated: {len(payments_agg)} rows")
print(payments_agg.head())

Payments aggregated: 99440 rows
                           order_id  total_payment
0  00010242fe8c5a6d1ba2dd792cb16214          72.19
1  00018f77f2f0320c557190d7a144bdd3         259.83
2  000229ec398224ef6ca0657da4fc703e         216.87
3  00024acbcdf0a6daa1e931b038114c75          25.78
4  00042b26cf59d7ce69dfabb4e55b4fd9         218.04


In [8]:
products_translated = products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

# Fixed: avoids the chained assignment warning
products_translated['product_category_name_english'] = products_translated['product_category_name_english'].fillna('unknown')

print(f"Products with categories: {len(products_translated)}")
print(products_translated[['product_id', 'product_category_name_english']].head())

Products with categories: 32951
                         product_id product_category_name_english
0  1e9e8ef04dbcff4541ed26657ea517e5                     perfumery
1  3aa071139cb16b67ca9e5dea641aaa2f                           art
2  96bd76ec8810374ed1b65e291975717f                sports_leisure
3  cef67bcfe19066a932b7673e239eb23d                          baby
4  9dc1a7de274444849c219cff195d0b71                    housewares


In [9]:
# Start with delivered orders
master = orders_delivered[[
    'order_id',
    'customer_id',
    'order_purchase_timestamp',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]].copy()

# Add customer state
master = master.merge(
    customers[['customer_id', 'customer_state', 'customer_city']],
    on='customer_id',
    how='left'
)

# Add order items (price, product_id, freight)
master = master.merge(
    order_items[['order_id', 'product_id', 'price', 'freight_value']],
    on='order_id',
    how='left'
)

# Add product category
master = master.merge(
    products_translated[['product_id', 'product_category_name_english']],
    on='product_id',
    how='left'
)

# Add total payment
master = master.merge(
    payments_agg,
    on='order_id',
    how='left'
)

print(f"Master table shape: {master.shape}")
print(master.head())

Master table shape: (110197, 12)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   
3  949d5b44dbf5de918fe9c16f97b45f8a  f88197465ea7920adcdbec7375364d82   
4  ad21c59c0840e6cb83a9ceb5573f8159  8ab97904e6daea8866dbdbc4fb7aad2c   

  order_purchase_timestamp order_delivered_customer_date  \
0      2017-10-02 10:56:33           2017-10-10 21:25:13   
1      2018-07-24 20:41:37           2018-08-07 15:27:45   
2      2018-08-08 08:38:49           2018-08-17 18:06:29   
3      2017-11-18 19:28:06           2017-12-02 00:28:42   
4      2018-02-13 21:18:39           2018-02-16 18:17:02   

  order_estimated_delivery_date customer_state            customer_city  \
0                    2017-10-18             SP                sao paulo   
1                

In [10]:
# Extract month and year for trend analysis
master['order_year'] = master['order_purchase_timestamp'].dt.year
master['order_month'] = master['order_purchase_timestamp'].dt.month
master['order_year_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)

# Calculate delivery time in days
master['delivery_days'] = (
    master['order_delivered_customer_date'] - master['order_purchase_timestamp']
).dt.days

# Calculate if delivery was on time
master['delivered_on_time'] = (
    master['order_delivered_customer_date'] <= master['order_estimated_delivery_date']
)

print("New columns added!")
print(master[['order_year_month', 'delivery_days', 'delivered_on_time']].head(10))

New columns added!
  order_year_month  delivery_days  delivered_on_time
0          2017-10            8.0               True
1          2018-07           13.0               True
2          2018-08            9.0               True
3          2017-11           13.0               True
4          2018-02            2.0               True
5          2017-07           16.0               True
6          2017-05            9.0               True
7          2017-01            9.0               True
8          2017-07           18.0               True
9          2017-05           12.0               True


In [11]:
print("=== MISSING VALUES IN MASTER TABLE ===")
print(master.isnull().sum())

# Drop rows where price is missing (can't analyze without it)
before = len(master)
master.dropna(subset=['price'], inplace=True)
after = len(master)
print(f"\nDropped {before - after} rows with missing price")
print(f"Final master table: {after} rows")

=== MISSING VALUES IN MASTER TABLE ===
order_id                         0
customer_id                      0
order_purchase_timestamp         0
order_delivered_customer_date    8
order_estimated_delivery_date    0
customer_state                   0
customer_city                    0
product_id                       0
price                            0
freight_value                    0
product_category_name_english    0
total_payment                    3
order_year                       0
order_month                      0
order_year_month                 0
delivery_days                    8
delivered_on_time                0
dtype: int64

Dropped 0 rows with missing price
Final master table: 110197 rows


In [12]:
print("=== FINAL MASTER TABLE SUMMARY ===")
print(f"Total rows:        {len(master)}")
print(f"Unique orders:     {master['order_id'].nunique()}")
print(f"Unique customers:  {master['customer_id'].nunique()}")
print(f"Unique categories: {master['product_category_name_english'].nunique()}")
print(f"Date range:        {master['order_purchase_timestamp'].min().date()} to {master['order_purchase_timestamp'].max().date()}")
print(f"Total revenue:     R$ {master['price'].sum():,.2f}")

=== FINAL MASTER TABLE SUMMARY ===
Total rows:        110197
Unique orders:     96478
Unique customers:  96478
Unique categories: 72
Date range:        2016-09-15 to 2018-08-29
Total revenue:     R$ 13,221,498.11


In [13]:
# Save master table
master.to_csv('../data/cleaned/master_orders.csv', index=False)

# Save a monthly summary (useful for Power BI)
monthly = master.groupby('order_year_month').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('price', 'sum'),
    avg_order_value=('price', 'mean')
).reset_index()
monthly.to_csv('../data/cleaned/monthly_summary.csv', index=False)

# Save category summary
category = master.groupby('product_category_name_english').agg(
    total_revenue=('price', 'sum'),
    total_orders=('order_id', 'nunique')
).reset_index().sort_values('total_revenue', ascending=False)
category.to_csv('../data/cleaned/category_summary.csv', index=False)

# Save state summary
state = master.groupby('customer_state').agg(
    total_revenue=('price', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_delivery_days=('delivery_days', 'mean')
).reset_index().sort_values('total_revenue', ascending=False)
state.to_csv('../data/cleaned/state_summary.csv', index=False)

print("All cleaned files saved to data/cleaned/!")

All cleaned files saved to data/cleaned/!
